# Fase 3 — MLP PyTorch: Treino, Comparação e Análise de Custo

Este notebook implementa e avalia a rede neural MLP (`ChurnMLP`) como modelo principal do projeto.

**Pré-requisito:** `notebooks/03_baseline.ipynb` deve ter sido executado antes deste,
pois a comparação de modelos consulta os runs de baseline já registrados no MLflow.

**Seções:**
1. Carregamento dos Dados
2. Configuração do MLflow
3. Treino do MLP v1 (ChurnMLP — PyTorch)
4. Comparação de Modelos — MLP vs Baselines
5. Análise de Custo — Threshold Ótimo (FP × R$50 | FN × R$500)
6. Resumo (Split 70/15/15)
7. Experimento Alternativo — Split 80/10/10
8. Avaliação Final no Conjunto de Teste Cego (80/10/10)

In [ ]:
from __future__ import annotations

import tempfile
import warnings

import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.artifacts
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch

from churn.config import (
    COST_FALSE_NEGATIVE,
    COST_FALSE_POSITIVE,
    MLFLOW_EXPERIMENT_NAME,
    ROC_AUC_TARGET,
    SEED,
)
from churn.data.loader import load_raw_data
from churn.data.preprocessing import (
    build_preprocessing_pipeline,
    clean_raw,
    split_features_target,
    stratified_split,
)
from churn.training.tracking import log_mlp_cv_run, setup_mlflow

warnings.filterwarnings("ignore")
torch.manual_seed(SEED)

## 1. Carregamento dos Dados

Mesmo pipeline determinístico usado no notebook de baselines (seed=42, split 70/15/15 estratificado).

In [ ]:
df_raw = load_raw_data()
df_clean = clean_raw(df_raw)
X, y = split_features_target(df_clean)
splits = stratified_split(X, y)

print(f"Treino: {splits.X_train.shape[0]:>5,} linhas | churn = {splits.y_train.mean():.1%}")
print(f"Val:    {splits.X_val.shape[0]:>5,} linhas | churn = {splits.y_val.mean():.1%}")
print(f"Teste:  {splits.X_test.shape[0]:>5,} linhas | churn = {splits.y_test.mean():.1%}")
print(f"\nFeatures (pré-preprocessamento): {splits.X_train.shape[1]}")

## 2. Configuração do MLflow

In [ ]:
setup_mlflow()
print(f"Experimento : {MLFLOW_EXPERIMENT_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 3. Treino do MLP v1 (ChurnMLP — PyTorch)

**Arquitetura** (CLAUDE.md §6 Fase 3):
```
Input → BatchNorm1d
      → Linear(n_features → 64) → ReLU → Dropout(0.3)
      → Linear(64 → 32)         → ReLU → Dropout(0.2)
      → Linear(32 → 1)  [logits]
```

**Treino:** Adam lr=1e-3 | BCEWithLogitsLoss com pos_weight automático | batch=64 | max_epochs=100 | patience=10

`log_mlp_cv_run` executa 5-fold CV estratificado + holdout refit e registra tudo no MLflow
(métricas, curvas por época, modelo serializado, matriz de confusão).

In [ ]:
print("Treinando MLP v1 — 5-fold CV + holdout refit...")
print("(pode levar alguns minutos na primeira execução)\n")

mlp_run_id = log_mlp_cv_run(
    model_name="mlp_v1",
    build_preprocessor=build_preprocessing_pipeline,
    X_train=splits.X_train,
    y_train=splits.y_train,
    X_val=splits.X_val,
    y_val=splits.y_val,
)

mlp_data = mlflow.get_run(mlp_run_id).data
print(f"Run ID      : {mlp_run_id}")
print(f"ROC-AUC CV  : {mlp_data.metrics['roc_auc_mean']:.4f} ± {mlp_data.metrics['roc_auc_std']:.4f}")
print(f"ROC-AUC hold: {mlp_data.metrics['holdout_val_roc_auc']:.4f}")
print(f"PR-AUC CV   : {mlp_data.metrics['pr_auc_mean']:.4f}")
print(f"F1 CV       : {mlp_data.metrics['f1_mean']:.4f}")
print(f"Best epoch  : {mlp_data.params['best_epoch']}")
print(f"Early stop  : {mlp_data.params['stopped_early']}")

beat_target = mlp_data.metrics['roc_auc_mean'] >= ROC_AUC_TARGET
print(f"\nAlvo ROC-AUC >= {ROC_AUC_TARGET}: {'ATINGIDO' if beat_target else 'NAO ATINGIDO'}")

## 4. Comparação de Modelos — MLP vs Baselines

Consulta todos os runs registrados no experimento e monta a tabela comparativa.
Referência principal: `logreg_baseline` com ROC-AUC (CV) = **0.8588 ± 0.0049**.

In [ ]:
KNOWN_RUN_NAMES = [
    "dummy_baseline",
    "logreg_baseline",
    "logreg_no_multilines_ablation",
    "logreg_no_phone_ablation",
    "logreg_no_phone_no_multilines_ablation",
    "mlp_v1",
]

runs_raw = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    order_by=["start_time DESC"],
)

col_map = {
    "tags.mlflow.runName": "modelo",
    "metrics.roc_auc_mean": "roc_auc_cv",
    "metrics.roc_auc_std": "roc_auc_std",
    "metrics.holdout_val_roc_auc": "roc_auc_holdout",
    "metrics.pr_auc_mean": "pr_auc_cv",
    "metrics.holdout_val_pr_auc": "pr_auc_holdout",
    "metrics.f1_mean": "f1_cv",
    "metrics.holdout_val_f1": "f1_holdout",
    "metrics.recall_mean": "recall_cv",
    "metrics.holdout_val_recall": "recall_holdout",
}

available_cols = [c for c in col_map if c in runs_raw.columns]
mask = runs_raw["tags.mlflow.runName"].isin(KNOWN_RUN_NAMES)

comparison = (
    runs_raw[mask][available_cols]
    .rename(columns={k: col_map[k] for k in available_cols})
    .drop_duplicates(subset="modelo", keep="first")
    .sort_values("roc_auc_cv", ascending=False)
    .reset_index(drop=True)
)

print(f"Runs encontrados: {len(comparison)} de {len(KNOWN_RUN_NAMES)} esperados")

In [ ]:
with pd.option_context(
    "display.float_format", "{:.4f}".format,
    "display.max_columns", 20,
    "display.width", 120,
):
    display(comparison)

In [ ]:
LOGREG_AUC_BASELINE = 0.8588

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

palette = [
    "#1B5E20" if "mlp" in m else "#90CAF9"
    for m in comparison["modelo"]
]

metric_configs = [
    ("roc_auc_cv", "ROC-AUC (CV mean)", ROC_AUC_TARGET, LOGREG_AUC_BASELINE),
    ("pr_auc_cv",  "PR-AUC  (CV mean)", 0.60,          None),
]

for ax, (metric, title, target, ref) in zip(axes, metric_configs):
    if metric not in comparison.columns:
        ax.set_visible(False)
        continue
    bars = ax.barh(
        comparison["modelo"],
        comparison[metric],
        color=palette,
        edgecolor="white",
        height=0.6,
    )
    ax.axvline(target, color="red", linestyle="--", linewidth=1.5, label=f"Alvo mínimo ({target:.2f})")
    if ref is not None:
        ax.axvline(ref, color="#FF8F00", linestyle=":", linewidth=1.8,
                   label=f"LogReg baseline ({ref:.4f})")
    ax.bar_label(bars, fmt="%.4f", padding=5, fontsize=9)
    ax.set_xlabel(metric, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_xlim(0, min(1.05, comparison[metric].max() + 0.08))

plt.suptitle("Comparação de Modelos — Churn Prediction", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Análise de Custo — Threshold Ótimo

O threshold padrão (0.5) não é o ótimo do ponto de vista de negócio. A relação de custo é:

| Erro | Significado | Custo |
|------|-------------|-------|
| **FP** | Cliente sem risco recebe ação de retenção | R\$ 50 |
| **FN** | Cliente churner não é abordado e cancela | R\$ 500 |

Custo FN é **10× maior** que FP → o threshold ótimo estará abaixo de 0.5
(classificar como churn mais agressivamente compensa o custo de ações desnecessárias).

In [ ]:
# Recarrega o modelo e o preprocessador do artefato MLflow
final_model = mlflow.pytorch.load_model(f"runs:/{mlp_run_id}/mlp_model")

with tempfile.TemporaryDirectory() as tmpdir:
    preproc_local = mlflow.artifacts.download_artifacts(
        artifact_uri=f"runs:/{mlp_run_id}/preprocessor.joblib",
        dst_path=tmpdir,
    )
    final_preprocessor = joblib.load(preproc_local)

# Probabilidades no conjunto de validação
X_val_t = np.asarray(final_preprocessor.transform(splits.X_val), dtype=np.float32)
final_model.eval()
with torch.no_grad():
    logits = final_model(torch.as_tensor(X_val_t))
    y_val_proba = torch.sigmoid(logits).numpy()

y_val_true = splits.y_val.to_numpy()
print(f"Val: {len(y_val_true)} amostras | taxa de churn real: {y_val_true.mean():.1%}")
print(f"Probabilidade média predita: {y_val_proba.mean():.4f}")

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)
costs, fp_counts, fn_counts = [], [], []

for thresh in thresholds:
    y_pred = (y_val_proba >= thresh).astype(int)
    fp = int(((y_pred == 1) & (y_val_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_val_true == 1)).sum())
    costs.append(fp * COST_FALSE_POSITIVE + fn * COST_FALSE_NEGATIVE)
    fp_counts.append(fp)
    fn_counts.append(fn)

costs = np.array(costs)
optimal_idx = int(np.argmin(costs))
optimal_threshold = float(thresholds[optimal_idx])
optimal_cost = float(costs[optimal_idx])

ref_idx = int(np.argmin(np.abs(thresholds - 0.5)))
ref_cost = float(costs[ref_idx])
cost_saving = ref_cost - optimal_cost

print(f"Threshold padrão  (0.50): custo estimado = R$ {ref_cost:>10,.0f}")
print(f"Threshold ótimo  ({optimal_threshold:.2f}): custo estimado = R$ {optimal_cost:>10,.0f}")
print(f"Economia estimada        : R$ {cost_saving:>10,.0f}  ({cost_saving / ref_cost:.1%} de redução)")
print(f"\nFP no threshold ótimo : {fp_counts[optimal_idx]}")
print(f"FN no threshold ótimo : {fn_counts[optimal_idx]}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Curva de custo total ---
ax1.plot(thresholds, costs, color="#1565C0", linewidth=2, label="Custo total")
ax1.fill_between(thresholds, costs, costs.min(), alpha=0.07, color="#1565C0")
ax1.axvline(
    optimal_threshold, color="#E53935", linestyle="--", linewidth=2,
    label=f"Ótimo = {optimal_threshold:.2f}  (R$ {optimal_cost:,.0f})",
)
ax1.axvline(
    0.5, color="gray", linestyle=":", linewidth=1.5,
    label=f"Padrão = 0.50  (R$ {ref_cost:,.0f})",
)
ax1.set_xlabel("Threshold", fontsize=11)
ax1.set_ylabel("Custo total estimado (R$)", fontsize=11)
ax1.set_title(
    f"Custo por Threshold — MLP v1\n"
    f"FP = R${COST_FALSE_POSITIVE:.0f}  |  FN = R${COST_FALSE_NEGATIVE:.0f}",
    fontsize=11, fontweight="bold",
)
ax1.legend(fontsize=9)
ax1.yaxis.set_major_formatter(lambda x, _: f"R${x:,.0f}")

# --- FP e FN por threshold ---
ax2.plot(thresholds, fp_counts, color="#FF8F00", linewidth=2, label="FP (ação desnecessária)")
ax2.plot(thresholds, fn_counts, color="#B71C1C", linewidth=2, label="FN (cliente perdido)")
ax2.axvline(
    optimal_threshold, color="#E53935", linestyle="--", linewidth=2,
    label=f"Threshold ótimo = {optimal_threshold:.2f}",
)
ax2.axvline(0.5, color="gray", linestyle=":", linewidth=1.5, label="Threshold padrão = 0.50")
ax2.set_xlabel("Threshold", fontsize=11)
ax2.set_ylabel("Contagem de erros", fontsize=11)
ax2.set_title("FP e FN por Threshold — MLP v1", fontsize=11, fontweight="bold")
ax2.legend(fontsize=9)

plt.suptitle("Análise de Custo — Threshold Ótimo", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("cost_analysis.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Registra o threshold ótimo e a análise de custo no mesmo run do MLflow
with mlflow.start_run(run_id=mlp_run_id):
    mlflow.log_params({
        "optimal_threshold": round(optimal_threshold, 4),
        "cost_fp": COST_FALSE_POSITIVE,
        "cost_fn": COST_FALSE_NEGATIVE,
    })
    mlflow.log_metrics({
        "optimal_threshold_cost": optimal_cost,
        "threshold_05_cost": ref_cost,
        "cost_saving_vs_05": cost_saving,
    })
    mlflow.log_artifact("cost_analysis.png")
    mlflow.log_artifact("model_comparison.png")

print("Artefatos e métricas de custo registrados no run MLflow.")
print(f"Run ID: {mlp_run_id}")

## 6. Resumo e Próximos Passos

In [ ]:
mlp_final = mlflow.get_run(mlp_run_id).data

logreg_row = comparison[comparison["modelo"] == "logreg_baseline"]
logreg_auc = logreg_row["roc_auc_cv"].values[0] if not logreg_row.empty else LOGREG_AUC_BASELINE

mlp_auc = mlp_final.metrics["roc_auc_mean"]
delta = mlp_auc - logreg_auc
status = "MLP SUPERA o baseline" if delta > 0 else "MLP NAO supera o baseline"

sep = "=" * 60
print(sep)
print("  RESUMO FASE 3 — MLP PyTorch vs Baseline")
print(sep)
print()
print("  MLP v1:")
print(f"    ROC-AUC CV      : {mlp_auc:.4f} ± {mlp_final.metrics['roc_auc_std']:.4f}")
print(f"    ROC-AUC holdout : {mlp_final.metrics['holdout_val_roc_auc']:.4f}")
print(f"    PR-AUC CV       : {mlp_final.metrics['pr_auc_mean']:.4f}")
print(f"    F1 CV           : {mlp_final.metrics['f1_mean']:.4f}")
print(f"    Recall CV       : {mlp_final.metrics['recall_mean']:.4f}")
print()
print("  Análise de custo:")
print(f"    Threshold ótimo : {optimal_threshold:.2f}")
print(f"    Custo estimado  : R$ {optimal_cost:,.0f}")
print(f"    Economia vs 0.5 : R$ {cost_saving:,.0f} ({cost_saving / ref_cost:.1%})")
print()
print(f"  LogReg baseline ROC-AUC : {logreg_auc:.4f}")
print(f"  Delta MLP vs LogReg     : {'+' if delta >= 0 else ''}{delta:.4f}")
print()
print(f"  Status: {status}")
print()
print(sep)
print("  Próximo passo: Fase 4 — FastAPI /predict endpoint")
print(sep)

## 7. Experimento Alternativo — Split 80/10/10

Adicionar mais dados de treino (+703 amostras, +14%) é uma das alavancas mais simples para
melhorar redes neurais pequenas sem mudar a arquitetura. Neste experimento o split é 80/10/10:
treino/val para early-stopping/teste cego — todos estratificados na mesma seed 42.

> **Nota:** o "holdout" reportado pelo MLflow (seção val do CV) coincide com o conjunto de
> early-stopping, tornando-o levemente otimista. O número verdadeiramente cego está na seção 8.

In [ ]:
splits_80 = stratified_split(X, y, test_size=0.10, val_size=0.10)

print(f"Treino (80%): {splits_80.X_train.shape[0]:>5,} linhas | churn = {splits_80.y_train.mean():.1%}")
print(f"Val    (10%): {splits_80.X_val.shape[0]:>5,} linhas | churn = {splits_80.y_val.mean():.1%}")
print(f"Teste  (10%): {splits_80.X_test.shape[0]:>5,} linhas | churn = {splits_80.y_test.mean():.1%}")

# Check for existing mlp_v1_split80 run (may have been trained in a prior session)
existing_split80 = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    filter_string="tags.mlflow.runName = 'mlp_v1_split80'",
    order_by=["start_time DESC"],
)

if len(existing_split80) > 0:
    mlp_run_id_80 = existing_split80.iloc[0]["run_id"]
    print(f"\nRun existente encontrado: {mlp_run_id_80}")
    print("Reutilizando run existente.")
else:
    print("\nTreinando MLP v1 com split 80/10/10...")
    mlp_run_id_80 = log_mlp_cv_run(
        model_name="mlp_v1_split80",
        build_preprocessor=build_preprocessing_pipeline,
        X_train=splits_80.X_train,
        y_train=splits_80.y_train,
        X_val=splits_80.X_val,
        y_val=splits_80.y_val,
    )

mlp80_data = mlflow.get_run(mlp_run_id_80).data
print(f"\nRun ID      : {mlp_run_id_80}")
print(f"ROC-AUC CV  : {mlp80_data.metrics['roc_auc_mean']:.4f} +/- {mlp80_data.metrics['roc_auc_std']:.4f}")
print(f"ROC-AUC hold: {mlp80_data.metrics['holdout_val_roc_auc']:.4f}  (val == early-stop; pode ser levemente otimista)")
print(f"PR-AUC CV   : {mlp80_data.metrics['pr_auc_mean']:.4f}")
print(f"F1 CV       : {mlp80_data.metrics['f1_mean']:.4f}")

## 8. Avaliação Final no Conjunto de Teste Cego (80/10/10)

O conjunto de teste separado no split 80/10/10 (≈705 amostras) **nunca foi usado** em
nenhuma etapa de treino nem de early-stopping. Esta é a avaliação desenviesada do modelo
`mlp_v1_split80` — o único número que fecha o argumento sem ambiguidade.

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Load model and preprocessor from MLflow (never touched X_test, only transform)
final_model_80 = mlflow.pytorch.load_model(f"runs:/{mlp_run_id_80}/mlp_model")

with tempfile.TemporaryDirectory() as tmpdir:
    preproc_local_80 = mlflow.artifacts.download_artifacts(
        artifact_uri=f"runs:/{mlp_run_id_80}/preprocessor.joblib",
        dst_path=tmpdir,
    )
    preprocessor_80 = joblib.load(preproc_local_80)

X_test_80_arr = np.asarray(preprocessor_80.transform(splits_80.X_test), dtype=np.float32)
y_test_80 = splits_80.y_test.to_numpy()

final_model_80.eval()
with torch.no_grad():
    logits_test = final_model_80(torch.as_tensor(X_test_80_arr))
    y_test_proba = torch.sigmoid(logits_test).numpy()

blind_roc_auc = float(roc_auc_score(y_test_80, y_test_proba))
blind_pr_auc  = float(average_precision_score(y_test_80, y_test_proba))
y_test_pred   = (y_test_proba >= 0.5).astype(int)
blind_f1      = float(f1_score(y_test_80, y_test_pred, zero_division=0))
blind_prec    = float(precision_score(y_test_80, y_test_pred, zero_division=0))
blind_recall  = float(recall_score(y_test_80, y_test_pred, zero_division=0))

holdout_val_80 = mlflow.get_run(mlp_run_id_80).data.metrics.get("holdout_val_roc_auc", float("nan"))
holdout_val_70 = mlflow.get_run(mlp_run_id).data.metrics.get("holdout_val_roc_auc", float("nan"))

sep = "=" * 57
print(sep)
print("  Avaliacao Cega --- mlp_v1_split80 no X_test (10%)")
print(sep)
print(f"  Amostras   : {len(y_test_80)}")
print(f"  Taxa churn : {y_test_80.mean():.1%}")
print()
print(f"  ROC-AUC    : {blind_roc_auc:.4f}  <-- numero limpo (nunca visto)")
print(f"  PR-AUC     : {blind_pr_auc:.4f}")
print(f"  F1         : {blind_f1:.4f}")
print(f"  Precision  : {blind_prec:.4f}")
print(f"  Recall     : {blind_recall:.4f}")
print()
print("  Comparacao:")
print(f"    LogReg baseline CV       : 0.8588")
print(f"    MLP 70/15/15 holdout     : {holdout_val_70:.4f}")
print(f"    MLP 80/10/10 val hold.   : {holdout_val_80:.4f}  (val == early-stop; leve otimismo)")
print(f"    MLP 80/10/10 teste cego  : {blind_roc_auc:.4f}  (desenviesado)")
print(sep)

In [ ]:
# Log blind-test metrics into the split-80 MLflow run
with mlflow.start_run(run_id=mlp_run_id_80):
    mlflow.log_metrics({
        "blind_test_roc_auc": blind_roc_auc,
        "blind_test_pr_auc": blind_pr_auc,
        "blind_test_f1": blind_f1,
        "blind_test_precision": blind_prec,
        "blind_test_recall": blind_recall,
    })

print(f"Metricas de teste cego registradas no run {mlp_run_id_80}.")